# 04 — First measured improvement: common-word removal

**Question:** do common function words distract BM25 from the clause the user needs?

At baseline k=5, 103 of 140 train questions find their contract but miss the answer span entirely. We will change only token filtering, keeping chunk size, overlap, BM25 parameters, threshold, and k fixed.

The hypothesis, primary objective, and guardrails are declared in `experiments/OBJECTIVE.md` and `experiments/objective.json` before candidate measurement. This is one iteration; accepted and rejected results are both retained. No LLM or optional retrieval feature is used.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from run_experiment import run_iteration
from retrieval import BM25Retriever, load_inputs, tokenize

objective = json.loads((ROOT / 'experiments/objective.json').read_text(encoding='utf-8'))
display(pd.Series(objective, name='Acceptance policy'))
print('Original:', tokenize('Which clause does not allow sublicensing?'))
print('Filtered:', tokenize('Which clause does not allow sublicensing?', True))

## Run one iteration

The command-line equivalent is `python run_experiment.py`. The runner evaluates the unchanged baseline, diagnoses train failures, evaluates the fixed candidate, applies the acceptance policy, and records both outcomes.

Each run creates a new result folder and appends a log entry. Rerunning iteration 1 verifies the same proposal; it does not count as another distinct improvement. Timing can vary. Both 140 answerable and 23 unanswerable train questions are included; test questions are never retrieved.

In [ ]:
output, reference, candidate, log = run_iteration(ROOT)
print('Saved:', output)
print('Before:', log['diagnosis_before'])
print('After:', log['diagnosis_after'])

## Compare measured results

Recall measures how much of the needed clause we found. Precision measures how much returned text is relevant. Five returned passages represent a fixed reviewer budget. Correct negative abstention is still a weakness of the baseline; a retrieval improvement alone does not solve it.

The 1-point recall target and guardrail tolerances are case-study engineering choices. Train gains are development evidence, not claims of generalization or statistical significance.

In [ ]:
tables = []
for label, result in [('baseline', reference), ('stopwords', candidate)]:
    frame = pd.DataFrame(result['answerable']['metrics'])
    frame.insert(0, 'approach', label)
    tables.append(frame)
comparison = pd.concat(tables, ignore_index=True)
display(comparison.style.format({'precision':'{:.2%}', 'recall':'{:.2%}', 'hit':'{:.2%}'}))
display(pd.DataFrame([
    dict(approach=label,
         false_abstention=result['answerable']['answerable_false_abstention_rate'],
         correct_abstention=result['negative']['correct_abstention_rate'],
         p95_ms=result['answerable']['latency_ms']['p95'])
    for label, result in [('baseline', reference), ('stopwords', candidate)]
]))
print('ACCEPTED' if log['decision']['accepted'] else 'REJECTED')
display(pd.Series(log['decision']['checks'], name='Passed'))
comparison.to_csv(output / 'comparison.csv', index=False)

## Try either runnable approach

Edit `question` and select `baseline` or `stopwords`. Both configurations remain available even if this experiment is rejected. Edit a separate config file for future experiments; retain the original baseline and all previous results.

In [ ]:
approach = 'stopwords'
_, documents, _ = load_inputs(ROOT)
config_name = {'baseline':'bm25_baseline', 'stopwords':'bm25_stopwords'}[approach]
config = json.loads((ROOT / 'configs' / f'{config_name}.json').read_text(encoding='utf-8'))
model = BM25Retriever(documents, config)
question = 'Under the Fox and Glu Mobile wireless content license, when does the term expire?'
hits = model.search(question, k=5)
if hits:
    display(pd.DataFrame(hits)[['score','file_path','start','end','text']])
else:
    print('No relevant passage found.')

## Explain this step

We tested one simple hypothesis: common-word removal might improve clause ranking. We compared both configurations under the same train evaluation and used a predeclared acceptance rule. Keeping rejected ideas prevents presenting only successful experiments.

**Business perspective:** better recall can reduce missed clauses; stable precision limits irrelevant reading. This change adds no API fees or external data transfer, but remains keyword-based and does not establish legal correctness. Local timing is not a production benchmark.

**Checkpoint:** one distinct improvement iteration is complete. At least four further iterations are still required by the brief. We will inspect this result before choosing the next change. Test performance, the working app/CLI, and LLM experiments remain for later steps.